# Centroid Debug: Why Arithmetic Mean Falls Outside the Intersection Region

**Probe**: `73.42.215.135` (Vanilla CBG / LP model)

This notebook traces through the exact computation to show why the arithmetic-mean centroid (red X on maps) can fall **outside** the yellow intersection region, while the Shapely area-weighted centroid (gray X) stays inside.

**Root causes**:
1. The 6 intersection vertices come from **spherical 3D geometry** (`circle_intersections()` in `helpers.py`), but the yellow region is drawn with **planar degree-offset** polygons (Shapely)
2. The arithmetic mean of unevenly-distributed boundary vertices can exit a concave region; an area-weighted centroid cannot

In [ ]:
"""Cell 1: Setup & imports — load data, fit LP models."""

import sys
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from functools import reduce
from shapely.geometry import Polygon as ShapelyPolygon, MultiPolygon, Point as ShapelyPoint, MultiPoint
from itertools import combinations

PROJECT_ROOT = Path().resolve().parent.parent.parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'scripts' / 'analysis' / 'cbg_feasibility'))

from scripts.utils.helpers import (
    haversine,
    circle_intersections,
    circle_preprocessing,
    polygon_centroid,
    get_middle_intersection,
    is_within_cirle,
    rtt_to_km,
)
from scripts.analysis.cbg_feasibility.rtt_model import (
    RTTDistanceModel,
    haversine_distance,
    THEORETICAL_SLOPE,
)

# Reuse helpers from evaluate_million_scale
from evaluate_million_scale import (
    load_data,
    fit_lp_models,
    _circle_to_shapely_polygon,
)

plt.style.use('seaborn-v0_8-whitegrid')

PROBE_IP = '73.42.215.135'

# Load data and fit LP models
df, df_asn = load_data()
lp_models = fit_lp_models(df_asn)

probe_data = df_asn[df_asn['src_ip'] == PROBE_IP].copy()
true_lat = probe_data['probe_latitude'].iloc[0]
true_lon = probe_data['probe_longitude'].iloc[0]
print(f"\nProbe: {PROBE_IP}")
print(f"True location: ({true_lat:.4f}, {true_lon:.4f})")
print(f"Measurements to {len(probe_data)} anchors")

## Cell 2: Build circles for probe — show which survive preprocessing and why

In [ ]:
"""Build LP-model circles and show preprocessing results."""

# Build all circles from LP models
all_circles = []
circle_info = []

for _, row in probe_data.iterrows():
    anchor_ip = row['dst_ip']
    rtt = row['min_rtt']
    if anchor_ip not in lp_models:
        continue
    model = lp_models[anchor_ip]
    if not model.fitted:
        continue
    d = model.predict_distance(rtt)
    if d is None or d <= 0:
        continue
    r = d / 6371  # radians
    circle_tuple = (model.anchor_lat, model.anchor_lon, rtt, d, r)
    all_circles.append(circle_tuple)
    circle_info.append({
        'anchor_ip': anchor_ip,
        'anchor_lat': model.anchor_lat,
        'anchor_lon': model.anchor_lon,
        'rtt_ms': rtt,
        'radius_km': d,
        'radius_rad': r,
    })

print(f"Total circles before preprocessing: {len(all_circles)}\n")

# Run preprocessing (removes circles that fully contain other circles)
circles_kept = circle_preprocessing(all_circles, speed_threshold=2/3)
kept_set = set(circles_kept)

# Build display table
rows = []
for i, (ct, ci) in enumerate(zip(all_circles, circle_info)):
    kept = ct in kept_set
    rows.append({
        'anchor': ci['anchor_ip'],
        'lat': f"{ci['anchor_lat']:.2f}",
        'lon': f"{ci['anchor_lon']:.2f}",
        'RTT (ms)': f"{ci['rtt_ms']:.2f}",
        'radius (km)': f"{ci['radius_km']:.0f}",
        'kept': 'YES' if kept else 'REMOVED',
    })

df_circles = pd.DataFrame(rows)
print(df_circles.to_string(index=False))
print(f"\nCircles surviving preprocessing: {len(circles_kept)} / {len(all_circles)}")

# Explain removals
removed = [(ct, ci) for ct, ci in zip(all_circles, circle_info) if ct not in kept_set]
if removed:
    print("\n--- Removal reasons (circle inclusion) ---")
    for ct, ci in removed:
        lat_r, lon_r, rtt_r, d_r, r_r = ct
        # Find which circle contains it
        for ct2, ci2 in zip(all_circles, circle_info):
            if ct2 == ct or ct2 not in kept_set:
                continue
            lat2, lon2, rtt2, d2, r2 = ct2
            dist_centers = haversine((lat_r, lon_r), (lat2, lon2))
            if d_r > (dist_centers + d2):
                print(f"  {ci['anchor_ip']} (r={d_r:.0f} km) fully contains "
                      f"{ci2['anchor_ip']} (r={d2:.0f} km, center dist={dist_centers:.0f} km)")

## Cell 3: Compute spherical intersection vertices

Run `circle_intersections()` on the surviving circles and display the vertices with the arithmetic mean calculation step by step.

In [ ]:
"""Compute spherical intersection vertices and arithmetic centroid."""

# Run the exact same pipeline as evaluate_million_scale.py
intersections, circles_out = circle_intersections(all_circles, speed_threshold=2/3)

print(f"Surviving circles after preprocessing: {len(circles_out)}")
print(f"Intersection vertices (filtered): {len(intersections)}\n")

# Display vertices
print("Intersection vertices (spherical geometry):")
print(f"{'Vertex':<8} {'Latitude':>10} {'Longitude':>12}")
print("-" * 32)
lat_sum, lon_sum = 0.0, 0.0
for i, (vlat, vlon) in enumerate(intersections):
    print(f"P{i+1:<7} {vlat:>10.4f} {vlon:>12.4f}")
    lat_sum += vlat
    lon_sum += vlon

n = len(intersections)
arith_lat = lat_sum / n
arith_lon = lon_sum / n

print(f"\n--- Arithmetic mean centroid ---")
print(f"  lat = ({lat_sum:.4f}) / {n} = {arith_lat:.4f}")
print(f"  lon = ({lon_sum:.4f}) / {n} = {arith_lon:.4f}")
print(f"  Centroid: ({arith_lat:.4f}, {arith_lon:.4f})")

# Verify against polygon_centroid()
pc = polygon_centroid(intersections)
print(f"\n  polygon_centroid() returns: ({pc[0]:.4f}, {pc[1]:.4f})")
assert abs(pc[0] - arith_lat) < 1e-6 and abs(pc[1] - arith_lon) < 1e-6, "Mismatch!"
print("  (matches — confirmed)")

# Also compute Shapely geometric centroid for comparison
circles_data_plot = [(lat, lon, d) for lat, lon, rtt, d, r in circles_out]
shapely_circles = [_circle_to_shapely_polygon(clat, clon, r_km) for clat, clon, r_km in circles_data_plot]
shapely_circles = [p for p in shapely_circles if p.is_valid and not p.is_empty]
intersection_poly = reduce(lambda a, b: a.intersection(b), shapely_circles)
shapely_c = intersection_poly.centroid
shapely_lat, shapely_lon = shapely_c.y, shapely_c.x

print(f"\n--- Shapely geometric centroid ---")
print(f"  Centroid: ({shapely_lat:.4f}, {shapely_lon:.4f})")

# Distances
arith_err = haversine((arith_lat, arith_lon), (true_lat, true_lon))
shapely_err = haversine((shapely_lat, shapely_lon), (true_lat, true_lon))
print(f"\n--- Error distances ---")
print(f"  Arithmetic centroid → true: {arith_err:.1f} km")
print(f"  Shapely centroid    → true: {shapely_err:.1f} km")
print(f"  Difference: {abs(arith_err - shapely_err):.1f} km")

## Cell 4: Scatter plot of vertices + centroids

Plot the spherical intersection vertices on a simple lat/lon scatter. Shows how the vertex distribution pulls the arithmetic mean away from the intersection region.

In [ ]:
"""Scatter plot of vertices + centroids."""

fig, ax = plt.subplots(figsize=(10, 7))

# Plot intersection vertices
v_lats = [p[0] for p in intersections]
v_lons = [p[1] for p in intersections]
ax.scatter(v_lons, v_lats, c='steelblue', s=80, zorder=5, edgecolors='black', linewidths=0.8)
for i, (vlat, vlon) in enumerate(intersections):
    ax.annotate(f'P{i+1}', (vlon, vlat), textcoords='offset points',
                xytext=(8, 5), fontsize=9, fontweight='bold', color='steelblue')

# Arithmetic mean centroid
ax.scatter([arith_lon], [arith_lat], c='red', s=200, marker='X', zorder=10,
           edgecolors='black', linewidths=1.2, label=f'Arith centroid ({arith_lat:.2f}, {arith_lon:.2f})')

# Shapely geometric centroid
ax.scatter([shapely_lon], [shapely_lat], c='gray', s=200, marker='X', zorder=10,
           edgecolors='black', linewidths=1.2, label=f'Geom centroid ({shapely_lat:.2f}, {shapely_lon:.2f})')

# True location
ax.scatter([true_lon], [true_lat], c='green', s=250, marker='*', zorder=10,
           edgecolors='black', linewidths=1, label=f'True ({true_lat:.2f}, {true_lon:.2f})')

# Draw Shapely intersection region outline
if not intersection_poly.is_empty:
    polys = list(intersection_poly.geoms) if isinstance(intersection_poly, MultiPolygon) else [intersection_poly]
    for poly in polys:
        xs, ys = poly.exterior.xy
        ax.fill(list(xs), list(ys), color='yellow', alpha=0.3, label='Intersection region')
        ax.plot(list(xs), list(ys), color='orange', linewidth=1.5)

# Annotate vertex clustering
ax.axvline(x=-126, color='gray', linestyle=':', alpha=0.5)
ax.text(-126, max(v_lats) + 0.5, 'lon = -126', ha='center', fontsize=8, color='gray')

ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title(f'Intersection Vertices & Centroids — Probe {PROBE_IP}', fontsize=13, fontweight='bold')
ax.legend(loc='best', fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Count clustering
n_west = sum(1 for lon in v_lons if lon < -126)
print(f"\nVertex clustering: {n_west}/{len(v_lons)} vertices have lon < -126°")
print("This westward cluster pulls the arithmetic mean to the left of the intersection region.")

## Cell 5: Map visualization

Cartopy map with the surviving circles, intersection region, spherical vertices, both centroids, and the true location.

In [ ]:
"""Map visualization with circles, intersection, vertices, and centroids."""

import cartopy.crs as ccrs
import cartopy.feature as cfeature

proj = ccrs.LambertConformal(central_longitude=-96, central_latitude=39)
fig, ax = plt.subplots(figsize=(14, 10), subplot_kw={'projection': proj})

ax.set_extent([-135, -66, 24, 55], crs=ccrs.PlateCarree())
ax.add_feature(cfeature.LAND, facecolor='#f0f0f0')
ax.add_feature(cfeature.OCEAN, facecolor='#e6f2ff')
ax.add_feature(cfeature.STATES, linewidth=0.5, edgecolor='gray')
ax.add_feature(cfeature.BORDERS, linewidth=0.8)
ax.coastlines(resolution='50m', linewidth=0.8)

colors = plt.cm.tab10(np.linspace(0, 1, max(len(circles_data_plot), 10)))

# Draw surviving circles
for i, (clat, clon, radius_km) in enumerate(circles_data_plot):
    n_pts = 100
    angles = np.linspace(0, 2 * np.pi, n_pts)
    r_deg_lat = radius_km / 111.0
    r_deg_lon = radius_km / (111.0 * math.cos(math.radians(clat)))
    circle_lons = clon + r_deg_lon * np.cos(angles)
    circle_lats = clat + r_deg_lat * np.sin(angles)
    ax.plot(circle_lons, circle_lats, color=colors[i], linewidth=1.5,
            alpha=0.7, transform=ccrs.PlateCarree())
    ax.fill(circle_lons, circle_lats, color=colors[i], alpha=0.05,
            transform=ccrs.PlateCarree())
    ax.plot(clon, clat, 's', color=colors[i], markersize=8,
            transform=ccrs.PlateCarree(), zorder=5,
            label=f'VP ({clat:.1f}, {clon:.1f}) r={radius_km:.0f}km')

# Draw Shapely intersection region (yellow)
if not intersection_poly.is_empty:
    polys = list(intersection_poly.geoms) if isinstance(intersection_poly, MultiPolygon) else [intersection_poly]
    for k, poly in enumerate(polys):
        if poly.is_empty or poly.geom_type != 'Polygon':
            continue
        xs, ys = poly.exterior.xy
        label = 'Intersection region' if k == 0 else None
        ax.fill(list(xs), list(ys), color='yellow', alpha=0.4,
                transform=ccrs.PlateCarree(), zorder=3, label=label)
        ax.plot(list(xs), list(ys), color='orange', linewidth=2,
                transform=ccrs.PlateCarree(), zorder=4)

# Spherical intersection vertices (gray dots)
ax.scatter(v_lons, v_lats, c='gray', s=40, marker='o', alpha=0.8,
           edgecolors='black', linewidths=0.5,
           transform=ccrs.PlateCarree(), zorder=6,
           label=f'Spherical vertices (n={len(intersections)})')

# Arithmetic centroid (red X) — this is what the code computes
ax.plot(arith_lon, arith_lat, 'X', color='red', markersize=16,
        markeredgecolor='black', markeredgewidth=1.2,
        transform=ccrs.PlateCarree(), zorder=10,
        label=f'Arith centroid (err={arith_err:.0f} km)')

# Shapely geometric centroid (gray X)
ax.plot(shapely_lon, shapely_lat, 'X', color='gray', markersize=16,
        markeredgecolor='black', markeredgewidth=1.2,
        transform=ccrs.PlateCarree(), zorder=10,
        label=f'Geom centroid (err={shapely_err:.0f} km)')

# True location (green star)
ax.plot(true_lon, true_lat, '*', color='green', markersize=20,
        markeredgecolor='black', markeredgewidth=1,
        transform=ccrs.PlateCarree(), zorder=10, label='True location')

# Annotation arrow from arith centroid to intersection region
ax.annotate('Outside intersection!',
            xy=(arith_lon, arith_lat), xycoords=ccrs.PlateCarree()._as_mpl_transform(ax),
            xytext=(30, 30), textcoords='offset points',
            fontsize=10, color='red', fontweight='bold',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.5))

ax.set_title(f'Centroid Debug — Probe {PROBE_IP}\n'
             f'Red X (arith) falls outside yellow region; Gray X (geometric) stays inside',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower left', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()

## Cell 6: Why they diverge — explanation

### Two geometry systems, two different centroids

The red X (arithmetic centroid) and gray X (Shapely geometric centroid) diverge because they are computed in **fundamentally different geometric spaces**:

| | Arithmetic Centroid (red X) | Shapely Geometric Centroid (gray X) |
|---|---|---|
| **Source geometry** | Spherical (3D Earth) | Planar (degree-offset ellipses) |
| **Intersection vertices** | `circle_intersections()` — exact pairwise intersections on a sphere | Shapely `.intersection()` — polygon clipping in lon/lat plane |
| **Centroid method** | Mean of vertex coordinates: $(\bar{\phi}, \bar{\lambda})$ | Area-weighted centroid of the 2D polygon |
| **Can exit the region?** | **Yes** — if vertices cluster unevenly on the boundary | **No** — guaranteed inside for convex polygons |

### Why the arithmetic mean exits the region

`polygon_centroid()` computes the simple arithmetic mean of the **spherical intersection vertices**:

$$\hat{\phi} = \frac{1}{n}\sum_{i=1}^{n}\phi_i, \quad \hat{\lambda} = \frac{1}{n}\sum_{i=1}^{n}\lambda_i$$

These vertices are **not uniformly distributed** around the intersection boundary — they are pairwise intersections of circles, so they cluster where circles are tangent or nearly tangent. When a majority of vertices cluster on one side of the region, the mean gets pulled toward that cluster and can exit the convex hull of the region (which itself is drawn in a different coordinate system).

### Why the Shapely centroid stays inside

Shapely's `.centroid` computes the area-weighted centroid of the polygon formed by intersecting the planar degree-offset circles. For a convex polygon, this is always inside the polygon by definition.

### Key insight

This is **expected behavior**, not a bug. The original Million-Scale paper uses the vertex-mean approach because it is simple and works well on average. The mismatch only becomes visible for specific probes where the vertex distribution is highly skewed.

## Cell 7: Quantitative comparison

Error distances, containment check, and vertex distribution statistics.

In [ ]:
"""Quantitative comparison: error distances and containment checks."""

print("=" * 60)
print("QUANTITATIVE COMPARISON")
print("=" * 60)

# Error distances
print(f"\n1. Error distances to true location ({true_lat:.4f}, {true_lon:.4f}):")
print(f"   Arithmetic centroid:  {arith_err:>8.1f} km")
print(f"   Shapely centroid:     {shapely_err:>8.1f} km")
print(f"   Difference:           {abs(arith_err - shapely_err):>8.1f} km")
better = "Shapely" if shapely_err < arith_err else "Arithmetic"
print(f"   Winner: {better}")

# Containment: is arith centroid inside the Shapely polygon?
arith_point = ShapelyPoint(arith_lon, arith_lat)
shapely_point = ShapelyPoint(shapely_lon, shapely_lat)

arith_inside = intersection_poly.contains(arith_point)
shapely_inside = intersection_poly.contains(shapely_point)
arith_on_boundary = intersection_poly.boundary.distance(arith_point) < 0.01

print(f"\n2. Containment in Shapely intersection polygon:")
print(f"   Arithmetic centroid inside:  {arith_inside}")
print(f"   Shapely centroid inside:     {shapely_inside}")
if not arith_inside:
    dist_to_boundary = intersection_poly.boundary.distance(arith_point)
    print(f"   Arith centroid distance to polygon boundary: {dist_to_boundary:.4f}°"
          f" (~{dist_to_boundary * 111:.1f} km)")

# Vertex distribution statistics
print(f"\n3. Vertex distribution:")
print(f"   N vertices: {len(intersections)}")
print(f"   Latitude  — min: {min(v_lats):.4f}, max: {max(v_lats):.4f}, "
      f"range: {max(v_lats)-min(v_lats):.4f}°")
print(f"   Longitude — min: {min(v_lons):.4f}, max: {max(v_lons):.4f}, "
      f"range: {max(v_lons)-min(v_lons):.4f}°")

# Check how many vertices are inside the Shapely polygon
n_inside = sum(1 for vlat, vlon in intersections
               if intersection_poly.buffer(0.1).contains(ShapelyPoint(vlon, vlat)))
print(f"\n4. Spherical vertices inside Shapely polygon (with 0.1° buffer): "
      f"{n_inside}/{len(intersections)}")
print("   (Vertices may fall outside the planar polygon because spherical and")
print("    planar circle boundaries don't exactly coincide)")

print("\n" + "=" * 60)
print("CONCLUSION: The arithmetic centroid falling outside the yellow")
print("intersection region is EXPECTED BEHAVIOR caused by the mismatch")
print("between spherical vertex computation and planar region drawing.")
print("=" * 60)